# Q-Mini English 512 — CPU-Only One-Click Resume Trainer (V12 — conversation data)

**पूरा ट्रेनिंग प्रोग्राम इसी notebook में है।** अलग Python files, pip commands, dataset upload या Termux की जरूरत नहीं। एक code cell में Drive authorization के बाद setup, data verification, model/checkpoint verification और training अपने-आप शुरू होते हैं।

## सिर्फ एक बार यह सेट करें
1. ऊपर **Runtime → Change runtime type → Hardware accelerator: None** चुनें और Save करें। यह CPU-only runtime है; T4/GPU न चुनें।
2. नीचे के एकमात्र code cell के बाईं ओर ▶ Run दबाएँ।
3. Google Drive अनुमति माँगे तो उसी Google account को चुनें जिसमें आपका Q-Mini Drive data/checkpoints हैं।

इसके बाद cell अपने-आप पूरा setup करता है: CPU lock, Drive mount, Python dependency install, pinned English conversation dataset preparation and verification (Drive cache reuse), Q-Mini source fetch, existing model/package verification, tokenizer/dataset reuse या preparation, latest verified checkpoint restore और CPU training. हर 10 optimizer steps पर verified checkpoint Drive में सेव होता है; runtime बंद होने पर अगली बार इसी notebook को दोबारा Run करने से resume होगा।

## पुराने GPU checkpoint से resume
V6 GPU auto-tuner batch size बदल सकता था। इस CPU notebook में batch 4 × gradient accumulation 2 रखा है और resume पर effective-batch-size परिवर्तन को स्पष्ट रूप से अनुमति दी गई है। Optimizer state रखा जाता है; tokens per optimizer step बदलने की चेतावनी output में दिखाई जाएगी। Data, tokenizer, model config और learning-rate contract की बाकी जाँचें लागू रहती हैं। अगर Drive में progress तो है लेकिन कोई valid checkpoint नहीं है, notebook random weights से दोबारा शुरू करने के बजाय रुक जाती है।

## ध्यान दें
- GPU device detect हुआ तो notebook training शुरू करने से पहले रुक जाएगी। इस तरह गलती से GPU runtime पर काम नहीं चलेगा।
- CPU training, T4 GPU से काफी धीमी होगी। 20,000-step लक्ष्य एक ही Colab session में पूरा न भी हो; verified Drive checkpoints के कारण बाद में resume किया जा सकता है।
- यह English conversation training है, final ChatGPT-जैसा assistant नहीं। Model quality का दावा केवल evaluation के बाद किया जा सकता है।
- पुराने training runtime को बंद करें; एक ही Drive checkpoint पर दो trainers एक साथ न चलाएँ।

In [ ]:
# Q-MINI ENGLISH 512 — ONE-CELL AUTO PRETRAINER
# Release marker: confirms that this fresh notebook, not an old Colab tab, is running.
print("NOTEBOOK BUILD: QMINI-ENGLISH-512-CPU-ONE-CLICK-RESUME-V12-2026-11-18")
# Uses the existing verified initial model. Does not rebuild random weights.
# Prepares pinned real human conversation data (DailyDialog + OpenAssistant oasst1); trains and saves
# resumable checkpoints and the final Q2-v2 model to Google Drive.

# Refuse a GPU-backed Colab runtime before hiding CUDA devices.
from pathlib import Path as _QMiniPath
if _QMiniPath('/dev/nvidiactl').exists() or _QMiniPath('/proc/driver/nvidia/version').exists():
    raise RuntimeError(
        'GPU hardware is attached to this Colab runtime. Open Runtime → Change runtime type, '
        'set Hardware accelerator to None (CPU), save, restart, and press Run again. '
        'No model, dataset, or checkpoint files have been changed.'
    )

# CPU-only lock: hide all GPUs BEFORE importing PyTorch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = str(min(4, max(1, os.cpu_count() or 2)))
os.environ["MKL_NUM_THREADS"] = os.environ["OMP_NUM_THREADS"]
os.environ["OPENBLAS_NUM_THREADS"] = os.environ["OMP_NUM_THREADS"]

from pathlib import Path
import torch

CPU_THREADS = min(4, max(1, os.cpu_count() or 2))
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

if torch.cuda.is_available():
    raise RuntimeError(
        "CPU-only lock did not take effect. Restart the Colab runtime and run this notebook again. "
        "No training or checkpoint files have been touched."
    )
print("CPU-only mode confirmed.")
print("CUDA_VISIBLE_DEVICES:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
print("PyTorch:", torch.__version__)
print("CPU threads:", torch.get_num_threads())
print("No GPU will be used by this notebook.")

from google.colab import drive
print("\nGoogle Drive authorization: approve access in the Google tab.")
print("When the page says 'Please close this tab / window', close only that authorization tab and return here.")
try:
    drive.mount("/content/drive", force_remount=False, timeout_ms=600000)
except Exception as drive_error:
    raise RuntimeError(
        "Google Drive did not mount. If the authorization tab showed 'Please close this tab / window', "
        "close that tab (not the Colab notebook) and return here. If this error repeats, share the full "
        "red traceback below; the training checkpoint and prepared data have not been touched."
    ) from drive_error
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive mount returned but MyDrive was not found; training will not start.")
print("Google Drive mount verified: /content/drive/MyDrive")

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid

# -------------------------- Fixed, verified settings --------------------------
REPOSITORY_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
EXPECTED_INITIAL_MODEL_REVISION = "b262787a81dd8297bd970624a55cff44b4424fe9"
TRAINING_SOURCE_REVISION = "da4df686b8d1073c43b26ad9e6e02c0278ac622c"
MODEL_CONFIG_REL = "configs/qmini_light_0.7m.json"

PACKAGE_NAME = "QMini-English-512-Q2v2-e1a187f0ca1567cd"
VOCAB_SIZE = 4096
VALIDATION_FRACTION = 0.02
TRAIN_SEQUENCE_LENGTH = 512
TARGET_OPTIMIZER_STEPS = 20000
BATCH_SIZE = 4
GRAD_ACCUM = 2
GLOBAL_BATCH_SIZE = 8
if BATCH_SIZE * GRAD_ACCUM != GLOBAL_BATCH_SIZE:
    raise RuntimeError("The optimized microbatch must preserve effective batch size 8.")
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY = 10
EVALUATE_EVERY = 100
EVALUATION_BATCHES = 8
CPU_THREAD_COUNT = CPU_THREADS
LEASE_STALE_SECONDS = 180
LEASE_HEARTBEAT_SECONDS = 45

def run_checked(command, *, cwd=None, capture_output=False, env=None):
    command = [str(item) for item in command]
    print("$", " ".join(command))
    # Always retain child output: notebook tracebacks otherwise hide the real
    # tokenizer/data exception behind a generic CalledProcessError.
    result = subprocess.run(
        command, cwd=str(cwd) if cwd else None, check=False,
        text=True, capture_output=True, env=env,
    )
    if result.stdout:
        print(result.stdout[-16000:])
    if result.returncode != 0:
        if result.stderr:
            print("----- subprocess stderr -----")
            print(result.stderr[-24000:])
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: "
            + " ".join(command)
            + "\\nSee the captured stdout/stderr immediately above."
        )
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

def download_verified(record, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = record["sha256"]
    if destination.is_file() and sha256_file(destination) == expected:
        print("Already downloaded and verified:", destination.name)
        return
    if destination.exists():
        destination.unlink()
    partial = destination.with_name(destination.name + ".part")
    partial.unlink(missing_ok=True)
    print("Downloading:", record["name"])
    request = Request(
        record["url"],
        headers={"User-Agent": "Q-Mini-Research-Training/1.0"},
    )
    total = 0
    with urlopen(request, timeout=90) as response, partial.open("wb") as output:
        while True:
            chunk = response.read(8 * 1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            total += len(chunk)
            if total and total % (32 * 1024 * 1024) < len(chunk):
                print("  received", f"{total / (1024**2):.1f} MiB")
    actual = sha256_file(partial)
    if actual != expected:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f"Dataset SHA-256 mismatch for {record['name']}; refusing to train on an unverified file."
        )
    partial.replace(destination)
    print("Verified:", destination.name, f"({total / (1024**2):.1f} MiB)")

print("\n[1/8] Checking CPU-only runtime...")
MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive is not connected. Approve the Drive prompt, return to Colab, then rerun.")

from packaging.version import Version
torch_public_version = torch.__version__.split("+", 1)[0]
if Version(torch_public_version) < Version("2.10.0"):
    raise RuntimeError(
        f"Q-Mini needs PyTorch >=2.10.0; this runtime has {torch.__version__}. "
        "Use a current Colab CPU runtime and rerun."
    )
if torch.cuda.is_available():
    raise RuntimeError("CPU-only safety check failed: CUDA became visible. No training was started.")
print("CPU-only preflight PASSED.")
print("PyTorch:", torch.__version__)
print("CPU thread count:", torch.get_num_threads())
try:
    import psutil
    _vm = psutil.virtual_memory()
    print(f"System RAM: {_vm.available / (1024**3):.2f} GiB available / {_vm.total / (1024**3):.2f} GiB total")
except Exception:
    print("System RAM monitor unavailable; continuing with the built-in memory preflight.")

# Preserve the Colab-provided PyTorch installation.
run_checked([
    sys.executable, "-m", "pip", "install", "--quiet",
    "numpy>=1.24", "tokenizers>=0.15", "pyarrow>=15.0", "packaging>=23",
])

print("\n[2/8] Checking Google Drive for a compatible initial model...")
INITIAL_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "initial_brains"
INITIAL_ROOT.mkdir(parents=True, exist_ok=True)
valid_initials = []
for folder in INITIAL_ROOT.glob("QMini-qmini_light_0.7m-Q2v2-initial-*"):
    manifest_path = folder / "model_manifest.json"
    config_path = folder / "config.json"
    model_path = folder / "model.qmini.npz"
    if not (manifest_path.is_file() and config_path.is_file() and model_path.is_file()):
        continue
    try:
        candidate_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        candidate_config = json.loads(config_path.read_text(encoding="utf-8"))
        if (
            candidate_manifest.get("format") == "qmini-initial-brain-v1"
            and candidate_manifest.get("status") == "untrained_initialization"
            and candidate_manifest.get("training_steps") == 0
            and candidate_manifest.get("optimizer_steps") == 0
            and candidate_manifest.get("source_revision") == EXPECTED_INITIAL_MODEL_REVISION
            and candidate_manifest.get("parameters") == 705742
            and candidate_config.get("max_context") == 512
            and candidate_config.get("vocab_size") == VOCAB_SIZE
            and candidate_manifest.get("packed_export_sha256") == sha256_file(model_path)
        ):
            valid_initials.append((folder.stat().st_mtime, folder, candidate_manifest))
    except (OSError, ValueError, TypeError):
        continue

SOURCE_REVISION = TRAINING_SOURCE_REVISION

print("\n[3/8] Fetching the exact Q-Mini source revision...")
REPO_DIR = Path("/content") / ("QMini-training-source-" + uuid.uuid4().hex[:8])
REPO_DIR.mkdir(parents=True, exist_ok=False)
run_checked(["git", "init"], cwd=REPO_DIR)
run_checked(["git", "remote", "add", "origin", REPOSITORY_URL], cwd=REPO_DIR)
run_checked(["git", "fetch", "--depth", "1", "origin", SOURCE_REVISION], cwd=REPO_DIR)
run_checked(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
actual_revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError("Downloaded Q-Mini source revision does not match the initial model manifest.")
if not (REPO_DIR / "qmini" / "cli.py").is_file():
    raise FileNotFoundError("The pinned Q-Mini source checkout is incomplete.")
sys.path.insert(0, str(REPO_DIR))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

# This kernel may have run an older notebook already. Remove cached Q-Mini
# modules so imports below definitely come from the newly pinned, fixed checkout.
for _loaded_module in tuple(sys.modules):
    if _loaded_module == "qmini" or _loaded_module.startswith("qmini."):
        del sys.modules[_loaded_module]


from qmini.config import QMiniConfig
from qmini.export import verify_packed_export
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS
from qmini.train import (
    _load_tokens,
    _validate_token_ids,
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
)

print("\\n[3.5/8] Running the SQLite thread-migration preflight before dataset work...")
from concurrent.futures import ThreadPoolExecutor
from qmini.data import iter_training_text_records

_preflight_dir = Path("/content") / ("qmini-thread-hop-" + uuid.uuid4().hex[:8])
_preflight_dir.mkdir(parents=True, exist_ok=False)
_preflight_corpus = _preflight_dir / "thread-hop.txt"
_preflight_iterator = None
try:
    _preflight_corpus.write_text(
        "\n\n".join(
            f"independent Q-Mini preflight document {index} contains enough unique words"
            for index in range(48)
        ),
        encoding="utf-8",
    )
    _preflight_iterator = iter_training_text_records(
        [_preflight_corpus], valid_fraction=0.02, near_duplicate_distance=0
    )
    _preflight_first = next(_preflight_iterator)
    with ThreadPoolExecutor(max_workers=1) as _executor:
        _preflight_second = _executor.submit(next, _preflight_iterator).result()
    _preflight_remaining = list(_preflight_iterator)
    if len({_preflight_first, _preflight_second, *_preflight_remaining}) < 2:
        raise RuntimeError("SQLite thread-migration preflight did not yield unique training records.")
    print(
        "SQLite thread-migration preflight PASSED:",
        2 + len(_preflight_remaining), "unique training records read across threads.",
    )
finally:
    if _preflight_iterator is not None:
        _preflight_iterator.close()
    shutil.rmtree(_preflight_dir, ignore_errors=True)

base_config = QMiniConfig.from_json(REPO_DIR / MODEL_CONFIG_REL)
if not valid_initials:
    print("No compatible initial brain found; building and verifying the step-zero Q2-v2 model now...")
    _initial_config = base_config
    if (
        _initial_config.max_context != TRAIN_SEQUENCE_LENGTH
        or _initial_config.vocab_size != VOCAB_SIZE
        or _initial_config.estimated_parameter_count() != 705742
    ):
        raise RuntimeError("Pinned Q-Mini config does not match the expected 705,742-parameter / 512-token English model.")
    _new_initial_dir = INITIAL_ROOT / (
        "QMini-qmini_light_0.7m-Q2v2-initial-"
        + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + "-" + uuid.uuid4().hex[:8]
    )
    from qmini.initial_build import build_initial_brain
    _new_initial_manifest = build_initial_brain(
        _initial_config, _new_initial_dir,
        source_revision=EXPECTED_INITIAL_MODEL_REVISION,
    )
    valid_initials.append((_new_initial_dir.stat().st_mtime, _new_initial_dir, _new_initial_manifest))

valid_initials.sort(key=lambda item: item[0], reverse=True)
_initial_time, INITIAL_DIR_SOURCE, INITIAL_MANIFEST = valid_initials[0]
INITIAL_MODEL_SOURCE = INITIAL_DIR_SOURCE / "model.qmini.npz"
INITIAL_SHA256 = INITIAL_MANIFEST["packed_export_sha256"]
INITIAL_MODEL_SOURCE_REVISION = INITIAL_MANIFEST.get("source_revision")
if INITIAL_MODEL_SOURCE_REVISION != EXPECTED_INITIAL_MODEL_REVISION:
    raise RuntimeError(
        "The saved initial model was not created from the pinned architecture revision. "
        f"Model revision={INITIAL_MODEL_SOURCE_REVISION!r}; "
        f"expected={EXPECTED_INITIAL_MODEL_REVISION!r}."
    )
print("Selected initial brain:", INITIAL_DIR_SOURCE.name)
print("Initial model SHA-256:", INITIAL_SHA256)


if base_config.max_context != TRAIN_SEQUENCE_LENGTH:
    raise RuntimeError("The GitHub model config does not have the required 512-token context.")
if base_config.estimated_parameter_count() != INITIAL_MANIFEST["parameters"]:
    raise RuntimeError("GitHub model config and saved initial model parameter count differ.")
initial_report = verify_packed_export(INITIAL_MODEL_SOURCE)
if (
    initial_report.get("format") != "qmini-packed-v2"
    or initial_report.get("q2_version") != 2
    or initial_report.get("parameter_count") != base_config.estimated_parameter_count()
):
    raise RuntimeError("The existing model failed Q2-v2 verification.")

print("\n[4/8] Preparing the real human conversation datasets...")
CONVERSATION_DATASETS_DIR = MY_DRIVE / "QMini-Light-0.7M" / "datasets"
DAILYDIALOG_DIR = CONVERSATION_DATASETS_DIR / "dailydialog"
OASST1_DIR = CONVERSATION_DATASETS_DIR / "oasst1_en"
PREPARE_SCRIPT = REPO_DIR / "scripts" / "prepare_conversation_datasets.py"
if not PREPARE_SCRIPT.is_file():
    raise RuntimeError(
        "The conversation preparation script is missing from the pinned source checkout; "
        "fetch a source revision that contains scripts/prepare_conversation_datasets.py."
    )
CONVERSATION_JSONL = [
    DAILYDIALOG_DIR / "dailydialog_train.jsonl",
    DAILYDIALOG_DIR / "dailydialog_validation.jsonl",
    DAILYDIALOG_DIR / "dailydialog_test.jsonl",
    OASST1_DIR / "oasst1_en_train.jsonl",
]
run_checked(
    [sys.executable, PREPARE_SCRIPT, "--datasets-dir", CONVERSATION_DATASETS_DIR],
    cwd=REPO_DIR,
)
for jsonl_path in CONVERSATION_JSONL:
    manifest_path = jsonl_path.parent / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}
    manifest_entry = next(
        (item for item in manifest.get("files", []) if item["path"] == jsonl_path.name),
        None,
    )
    if (
        not jsonl_path.is_file()
        or jsonl_path.stat().st_size == 0
        or manifest.get("format") != "qmini-conversation-dataset-v1"
        or manifest_entry is None
        or jsonl_path.stat().st_size != manifest_entry["size_bytes"]
        or sha256_file(jsonl_path) != manifest_entry["sha256"]
    ):
        raise RuntimeError(f"Conversation file failed verification: {jsonl_path}")
    with jsonl_path.open(encoding="utf-8") as handle:
        first = json.loads(handle.readline())
    if not isinstance(first.get("messages"), list) or not first["messages"]:
        raise RuntimeError(f"Conversation file has no chat messages: {jsonl_path}")
    if {message.get("role") for message in first["messages"]} - {"user", "assistant"}:
        raise RuntimeError(f"Conversation file has an unexpected role: {jsonl_path}")
data_fingerprints = [
    {"name": path.name, "sha256": sha256_file(path)} for path in CONVERSATION_JSONL
]
PACKAGE_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "training_ready"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_DIR = PACKAGE_ROOT / PACKAGE_NAME
STAGE_DIR = PACKAGE_ROOT / (".resume-" + PACKAGE_NAME)

print("Conversation files:", len(data_fingerprints))
print("Training package:", PACKAGE_DIR)

def verify_training_package(package_dir):
    package_dir = Path(package_dir)
    initial_dir = package_dir / "initial_brain"
    tokenizer_path = package_dir / "tokenizer" / "tokenizer.json"
    dataset_dir = package_dir / "dataset"
    runtime_dir = package_dir / "training_runtime"
    runtime_manifest_path = runtime_dir / "TRAINING_RUNTIME.json"
    config_path = initial_dir / "config.json"
    model_path = initial_dir / "model.qmini.npz"
    manifest_path = initial_dir / "model_manifest.json"
    progress_path = package_dir / "training_state" / "training_progress.json"
    required = [
        config_path, model_path, manifest_path, tokenizer_path,
        Path(str(tokenizer_path) + ".qmini.json"),
        package_dir / "active_config.json",
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        dataset_dir / "train.mask.bin", dataset_dir / "valid.mask.bin",
        dataset_dir / "dataset.json",
        runtime_manifest_path, runtime_dir / "qmini" / "train.py",
        runtime_dir / "qmini" / "checkpointing.py",
        progress_path,
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Training package is incomplete: " + ", ".join(missing))

    config = QMiniConfig.from_json(config_path)
    active_config = QMiniConfig.from_json(package_dir / "active_config.json")
    if config.to_dict() != active_config.to_dict():
        raise RuntimeError("Tokenizer changed the model architecture or vocabulary unexpectedly.")
    if config.max_context != 512 or config.vocab_size != VOCAB_SIZE:
        raise RuntimeError("Training package does not preserve 512 context and 4096 vocabulary.")
    if config.estimated_parameter_count() != 705742:
        raise RuntimeError("Training package parameter count changed unexpectedly.")
    loaded_tokenizer = load_tokenizer(tokenizer_path)
    if loaded_tokenizer.get_vocab_size() != VOCAB_SIZE:
        raise RuntimeError("Tokenizer vocabulary does not match the initial model.")
    for token in SPECIAL_TOKENS:
        if loaded_tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Required structural token is missing: {token}")

    _verify_initial_brain_artifact(model_path, config)
    _verify_prepared_dataset_manifest(
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        tokenizer_path, dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", config.vocab_size,
    )
    train_array = _load_tokens(dataset_dir / "train.bin", "Training")
    valid_array = _load_tokens(dataset_dir / "valid.bin", "Validation")
    _validate_token_ids(train_array, config.vocab_size, "Training")
    _validate_token_ids(valid_array, config.vocab_size, "Validation")
    if len(train_array) < 514 or len(valid_array) < 514:
        raise RuntimeError("Prepared data is too small for 512-token training sequences.")
    if (dataset_dir / "train.mask.bin").stat().st_size != (dataset_dir / "train.bin").stat().st_size // 2:
        raise RuntimeError("Training mask size does not match the token stream.")
    if (dataset_dir / "valid.mask.bin").stat().st_size != (dataset_dir / "valid.bin").stat().st_size // 2:
        raise RuntimeError("Validation mask size does not match the token stream.")

    runtime_manifest = json.loads(runtime_manifest_path.read_text(encoding="utf-8"))
    if runtime_manifest.get("enabled") is not False or runtime_manifest.get("requires_manual_activation") is not True:
        raise RuntimeError("Training runtime was unexpectedly enabled in the package.")
    for record in runtime_manifest.get("source_files", []):
        relative = Path(record["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Unsafe path in training runtime manifest.")
        source = runtime_dir / relative
        if not source.is_file() or sha256_file(source) != record["sha256"]:
            raise RuntimeError(f"Training source hash mismatch: {relative}")
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if model_manifest.get("training_runtime_manifest_sha256") != sha256_file(runtime_manifest_path):
        raise RuntimeError("Training runtime manifest hash does not match model manifest.")
    if model_manifest.get("packed_export_sha256") != sha256_file(model_path):
        raise RuntimeError("Initial model fingerprint changed inside the training package.")
    return config, model_manifest, train_array, valid_array

if PACKAGE_DIR.is_dir():
    print("\nExisting training package found; verifying and reusing it.")
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)

    # Refresh the disabled, hash-checked source snapshot to the fixed training
    # revision while retaining the already prepared tokenizer and dataset.
    RUNTIME_DIR = PACKAGE_DIR / "training_runtime"
    shutil.rmtree(RUNTIME_DIR / "qmini", ignore_errors=True)
    shutil.copytree(
        REPO_DIR / "qmini",
        RUNTIME_DIR / "qmini",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),
    )
    shutil.copy2(REPO_DIR / "pyproject.toml", RUNTIME_DIR / "pyproject.toml")
    _runtime_files = []
    for _source in sorted(
        p for p in RUNTIME_DIR.rglob("*")
        if p.is_file() and p.name != "TRAINING_RUNTIME.json"
    ):
        _runtime_files.append({
            "path": _source.relative_to(RUNTIME_DIR).as_posix(),
            "sha256": sha256_file(_source),
            "size_bytes": _source.stat().st_size,
        })
    write_json(RUNTIME_DIR / "TRAINING_RUNTIME.json", {
        "format": "qmini-training-runtime-v1",
        "enabled": False,
        "requires_manual_activation": True,
        "entrypoint": "qmini.train.train_model",
        "checkpoint_backend": "qmini.checkpointing",
        "source_revision": SOURCE_REVISION,
        "source_files": _runtime_files,
    })
    _initial_manifest_path = PACKAGE_DIR / "initial_brain" / "model_manifest.json"
    _initial_manifest = json.loads(_initial_manifest_path.read_text(encoding="utf-8"))
    _initial_manifest["training_runtime_manifest_sha256"] = sha256_file(
        RUNTIME_DIR / "TRAINING_RUNTIME.json"
    )
    write_json(_initial_manifest_path, _initial_manifest)

    _project_source = PACKAGE_DIR / "project_source"
    shutil.rmtree(_project_source, ignore_errors=True)
    shutil.copytree(
        REPO_DIR, _project_source,
        ignore=shutil.ignore_patterns(
            ".git", "__pycache__", "*.pyc", ".pytest_cache",
            ".venv", "venv", ".mypy_cache", ".ruff_cache",
        ),
    )
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
    print("Updated training-code snapshot; original tokenizer and prepared dataset were preserved.")
    # Switch the prepared token streams to the conversation data comma, keeping the
    # existing package, tokenizer, initial brain, checkpoint and progress.
    # The previous streams are archived, never deleted.
    package_dataset_dir = PACKAGE_DIR / "dataset"
    package_manifest_text = (package_dataset_dir / "dataset.json").read_text(encoding="utf-8")
    if not all(name in package_manifest_text for name in {path.name for path in CONVERSATION_JSONL}):
        archive_dir = PACKAGE_DIR / "dataset-previous-v1"
        if archive_dir.exists():
            raise RuntimeError(
                "A previous dataset archive already exists; refusing to switch "
                "the prepared dataset again until it is inspected."
            )
        package_dataset_dir.rename(archive_dir)
        print("Archived the previous prepared streams at:", archive_dir)
        conversation_out_dir = Path("/content") / ("qmini-conversation-dataset-" + PACKAGE_NAME)
        shutil.rmtree(conversation_out_dir, ignore_errors=True)
        conversation_out_dir.mkdir(parents=True, exist_ok=False)
        run_checked(
            [
                sys.executable, "-m", "qmini.cli", "prepare",
                "--data-dir", *[str(path) for path in CONVERSATION_JSONL],
                "--tokenizer", PACKAGE_DIR / "tokenizer" / "tokenizer.json",
                "--output-dir", conversation_out_dir,
                "--valid-fraction", str(VALIDATION_FRACTION),
                "--near-duplicate-hamming-distance", "0",
                "--loss-mask", "assistant",
            ],
            cwd=REPO_DIR,
        )
        _conversation_tmp = PACKAGE_DIR / (".dataset-copy-" + uuid.uuid4().hex[:8])
        shutil.copytree(conversation_out_dir, _conversation_tmp)
        _conversation_tmp.rename(package_dataset_dir)
        shutil.rmtree(conversation_out_dir, ignore_errors=True)
        _verify_prepared_dataset_manifest(
            package_dataset_dir / "train.bin",
            package_dataset_dir / "valid.bin",
            PACKAGE_DIR / "tokenizer" / "tokenizer.json",
            package_dataset_dir / "train.mask.bin",
            package_dataset_dir / "valid.mask.bin",
            VOCAB_SIZE,
        )
        print("Prepared conversation token streams and verified the package dataset.")
else:
    raise RuntimeError(
        "The existing training package QMini-English-512-Q2v2-e1a187f0ca1567cd "
        "was not found in Drive. This notebook only resumes that package "
        "from checkpoint-step-000000012540; it never builds a new package."
    )

# Re-verify persistent package every time before creating an optimizer.
PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
INITIAL_DIR = PACKAGE_DIR / "initial_brain"
INITIAL_MODEL_PATH = INITIAL_DIR / "model.qmini.npz"
CONFIG_PATH = INITIAL_DIR / "config.json"
TOKENIZER_PATH = PACKAGE_DIR / "tokenizer" / "tokenizer.json"
DATASET_DIR = PACKAGE_DIR / "dataset"
TRAIN_BIN = DATASET_DIR / "train.bin"
VALID_BIN = DATASET_DIR / "valid.bin"
TRAIN_MASK = DATASET_DIR / "train.mask.bin"
VALID_MASK = DATASET_DIR / "valid.mask.bin"
RUNTIME_DIR = PACKAGE_DIR / "training_runtime"
MIRROR_DIR = PACKAGE_DIR / "training_state"
PROGRESS_PATH = MIRROR_DIR / "training_progress.json"
LOCAL_RUN_DIR = Path("/content/qmini-english-512-runs") / PACKAGE_NAME

from qmini.train import (
    _load_tokens,
    _validate_token_ids,
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
    train_model,
)
from qmini.checkpointing import (
    TrainingSessionLease,
    mark_training_status,
    restore_latest_checkpoint,
)
from qmini.export import verify_packed_export

print("\n[6/8] Checking package, disk space and checkpoint safety...")
assert PACKAGE_CONFIG.max_context == 512
assert PACKAGE_CONFIG.estimated_parameter_count() == 705742
_verify_initial_brain_artifact(INITIAL_MODEL_PATH, PACKAGE_CONFIG)
_verify_prepared_dataset_manifest(
    TRAIN_BIN, VALID_BIN, TOKENIZER_PATH, TRAIN_MASK, VALID_MASK, PACKAGE_CONFIG.vocab_size
)
checkpoint_estimate_bytes = int(PACKAGE_CONFIG.estimated_parameter_count() * 12 * 1.2 + 512 * 1024**2)
drive_required_bytes = 3 * checkpoint_estimate_bytes + 1024**3
local_required_bytes = 2 * checkpoint_estimate_bytes + 1024**3
drive_free_bytes = shutil.disk_usage(MY_DRIVE).free
local_free_bytes = shutil.disk_usage("/content").free
print(f"Drive free: {drive_free_bytes / (1024**3):.2f} GiB")
print(f"Colab local free: {local_free_bytes / (1024**3):.2f} GiB")
if drive_free_bytes < drive_required_bytes:
    raise RuntimeError("Not enough Google Drive space for verified checkpoint generations and model export.")
if local_free_bytes < local_required_bytes:
    raise RuntimeError("Not enough Colab temporary disk space for safe checkpoint creation.")

progress_now = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
print("Current saved training status:", progress_now.get("status"))
print("Last verified step:", progress_now.get("current_step", 0), "/", progress_now.get("target_steps"))
print("Train tokens:", f"{len(train_tokens):,}")
print("Validation tokens:", f"{len(valid_tokens):,}")
print("Training device: CPU-only")

print("CPU threads:", CPU_THREADS)
print("Sequence length:", TRAIN_SEQUENCE_LENGTH)
print("Target optimizer steps:", TARGET_OPTIMIZER_STEPS)
print("CPU batch settings: physical batch=4 | gradient accumulation=2")

print("Effective batch=8 | tokens per optimizer step=4,096")

print("CPU mode: FP32; CUDA disabled.")


print("\n[7/8] Training/resuming the existing initial model...")
progress_now = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
if progress_now.get("status") == "completed" and progress_now.get("final_model_sha256"):
    print("This exact English dataset/model package is already fully trained. Verifying the saved final model; no duplicate training run will be started.")
    completed_step = progress_now.get("current_step")
    completed_target = progress_now.get("target_steps")
    if (
        isinstance(completed_step, bool)
        or not isinstance(completed_step, int)
        or completed_step < 1
        or completed_step != completed_target
    ):
        raise RuntimeError("Drive says training is complete, but the saved optimizer step/target is inconsistent.")
    final_drive_dir = (
        PACKAGE_DIR / "trained_models"
        / f"QMini-English-512-trained-step-{completed_step:012d}"
    )
    final_manifest_path = final_drive_dir / "model_manifest.json"
    if not final_manifest_path.is_file():
        raise FileNotFoundError("Drive progress says training is complete, but the verified final model manifest is missing.")
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
    if final_manifest.get("step") != completed_step:
        raise RuntimeError("Saved final model step does not match the completed Drive progress record.")
    final_packed_path = final_drive_dir / final_manifest["packed_export"]
    final_report = verify_packed_export(final_packed_path)
    if (
        final_report.get("format") != "qmini-packed-v2"
        or final_report.get("q2_version") != 2
        or final_report.get("parameter_count") != PACKAGE_CONFIG.estimated_parameter_count()
        or sha256_file(final_packed_path) != final_manifest.get("packed_export_sha256")
    ):
        raise RuntimeError("The previously completed final model failed Q2-v2 or SHA-256 verification.")
    _completed_digest = hashlib.sha256()
    for _item in sorted(final_drive_dir.rglob("*")):
        if _item.is_file():
            _completed_digest.update(_item.relative_to(final_drive_dir).as_posix().encode("utf-8"))
            _completed_digest.update(sha256_file(_item).encode("ascii"))
    if _completed_digest.hexdigest() != progress_now.get("final_model_sha256"):
        raise RuntimeError("The final model directory hash differs from the completed Drive progress record.")
    final_progress = progress_now
    print("Previously completed model passed final format and hash verification.")
else:
    LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
    session_started = datetime.now(timezone.utc).isoformat()
    with TrainingSessionLease(
        MIRROR_DIR,
        stale_after_seconds=LEASE_STALE_SECONDS,
        heartbeat_seconds=LEASE_HEARTBEAT_SECONDS,
    ) as lease:
        previous_progress = (
            json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
            if PROGRESS_PATH.is_file() else {}
        )
        if previous_progress.get("status") == "running":
            mark_training_status(
                MIRROR_DIR, "interrupted",
                note="Previous Colab runtime ended without releasing its training lease.",
            )

        restored = restore_latest_checkpoint(MIRROR_DIR, LOCAL_RUN_DIR)
        if restored is None:
            current_progress = (
                json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
                if PROGRESS_PATH.is_file() else {}
            )
            if current_progress.get("current_step", 0) not in (0, None):
                raise RuntimeError(
                    "Drive progress shows training steps, but no verified checkpoint can be restored. "
                    "Refusing to restart from random weights."
                )
            resume_run = False
            extend_schedule = False
            current_step = 0
        else:
            resume_run = True
            extend_schedule = TARGET_OPTIMIZER_STEPS > restored["target_steps"]
            current_step = restored["step"]
            if current_step > TARGET_OPTIMIZER_STEPS:
                raise RuntimeError("Saved optimizer step is above this notebook's target.")
            if restored["target_steps"] > TARGET_OPTIMIZER_STEPS and current_step < TARGET_OPTIMIZER_STEPS:
                raise RuntimeError("The new target is below the previous run's target; refusing to shorten the schedule.")

        # Keep the existing checkpoint's effective batch size unchanged.
        BATCH_SIZE = 4
        GRAD_ACCUM = 2
        GLOBAL_BATCH_SIZE = 8
        print("CPU training settings: batch=4, grad_accum=2, effective_batch=8")
        print("Tokens per optimizer step:", BATCH_SIZE * GRAD_ACCUM * TRAIN_SEQUENCE_LENGTH)

        mark_training_status(
            MIRROR_DIR, "running",
            note=f"Q-Mini English 512 training session started at {session_started}.",
        )
        print("Resume:", resume_run)
        print("Starting step:", current_step, "/", TARGET_OPTIMIZER_STEPS)

        print("Starting CPU-only training; CUDA is disabled and no GPU is used.")

        # Resume on the conversation data with the audited dataset switch.
        FINAL_LOCAL_DIR = train_model(
            config=PACKAGE_CONFIG,
            train_bin=TRAIN_BIN,
            valid_bin=VALID_BIN,
            output_dir=LOCAL_RUN_DIR,
            steps=TARGET_OPTIMIZER_STEPS,
            batch_size=BATCH_SIZE,
            seq_len=TRAIN_SEQUENCE_LENGTH,
            grad_accum=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            eval_every=EVALUATE_EVERY,
            save_every=CHECKPOINT_EVERY,
            eval_batches=EVALUATION_BATCHES,
            resume=resume_run,
            tokenizer_path=TOKENIZER_PATH,
            extend_schedule=extend_schedule,
            train_mask_bin=TRAIN_MASK,
            valid_mask_bin=VALID_MASK,
            init_from_initial=None if resume_run else INITIAL_MODEL_PATH,
            checkpoint_mirror_dir=MIRROR_DIR,
            checkpoint_session_id=lease.session_id,
            allow_batch_size_change=True,
            allow_dataset_change=True,
        )

        final_manifest_path = FINAL_LOCAL_DIR / "model_manifest.json"
        if not final_manifest_path.is_file():
            raise FileNotFoundError("Training did not produce a final_model manifest.")
        final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
        final_packed_path = FINAL_LOCAL_DIR / final_manifest["packed_export"]
        final_report = verify_packed_export(final_packed_path)
        if (
            final_report.get("format") != "qmini-packed-v2"
            or final_report.get("q2_version") != 2
            or final_report.get("parameter_count") != PACKAGE_CONFIG.estimated_parameter_count()
        ):
            raise RuntimeError("Final trained Q2-v2 export failed format/parameter verification.")
        if sha256_file(final_packed_path) != final_manifest.get("packed_export_sha256"):
            raise RuntimeError("Final trained model SHA-256 differs from its manifest.")

        final_step = int(final_manifest["step"])
        trained_root = PACKAGE_DIR / "trained_models"
        trained_root.mkdir(parents=True, exist_ok=True)
        final_name = f"QMini-English-512-trained-step-{final_step:012d}"
        final_drive_dir = trained_root / final_name
        staging_dir = trained_root / ("." + final_name + ".staging-" + uuid.uuid4().hex[:8])

        if final_drive_dir.exists():
            existing_manifest = json.loads((final_drive_dir / "model_manifest.json").read_text(encoding="utf-8"))
            existing_packed = final_drive_dir / existing_manifest["packed_export"]
            if (
                existing_manifest.get("step") != final_step
                or existing_manifest.get("packed_export_sha256") != final_manifest.get("packed_export_sha256")
                or sha256_file(existing_packed) != existing_manifest.get("packed_export_sha256")
            ):
                raise FileExistsError("An existing trained model conflicts with this final optimizer step.")
            verify_packed_export(existing_packed)
        else:
            try:
                shutil.copytree(FINAL_LOCAL_DIR, staging_dir)
                copied_manifest = json.loads((staging_dir / "model_manifest.json").read_text(encoding="utf-8"))
                copied_packed = staging_dir / copied_manifest["packed_export"]
                if sha256_file(copied_packed) != copied_manifest.get("packed_export_sha256"):
                    raise RuntimeError("Final model Drive-copy SHA-256 verification failed.")
                verify_packed_export(copied_packed)
                staging_dir.rename(final_drive_dir)
            except Exception:
                shutil.rmtree(staging_dir, ignore_errors=True)
                raise

        directory_digest = hashlib.sha256()
        for item in sorted(final_drive_dir.rglob("*")):
            if item.is_file():
                directory_digest.update(item.relative_to(final_drive_dir).as_posix().encode("utf-8"))
                directory_digest.update(sha256_file(item).encode("ascii"))
        final_model_digest = directory_digest.hexdigest()
        final_progress = mark_training_status(
            MIRROR_DIR, "completed",
            final_model_sha256=final_model_digest,
            note=f"Verified trained Q2-v2 model stored at trained_models/{final_name}",
        )

    print("\n[8/8] Q-MINI ENGLISH TRAINING COMPLETE")
    print("Optimizer step:", final_progress["current_step"], "/", final_progress["target_steps"])
    print("Tokens seen:", f"{final_progress['tokens_seen']:,}")
    print("Trained model:", final_drive_dir)
    print("Model file:", final_drive_dir / "model.qmini.npz")
    print("Verified checkpoints:", MIRROR_DIR)
    print("Progress file:", PROGRESS_PATH)
    print("\nTraining updates were made to the saved initial model. The original initial model remains unchanged.")
